# Personal Loan Default Risk: Decision Tree


This notebook trains a Decision Tree using Credit Score, DTI Ratio (%), and Savings ($) to predict Loan Status: 0 = Approved / Low Risk and 1 = Default / High Risk.



The local CSV uploader accepts the same three feature columns in the same order. Loan Status is optional for unlabeled prediction.


In [3]:
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

customers = pd.DataFrame({
    'Customer ID': ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10'],
    'Credit Score': [780, 720, 690, 660, 640, 610, 580, 550, 520, 480],
    'DTI Ratio (%)': [12, 18, 25, 32, 38, 42, 48, 55, 60, 68],
    'Savings ($)': [25000, 15000, 8000, 5000, 3000, 2000, 1200, 800, 400, 100],
    'Loan Status': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
})
feature_columns = ['Credit Score', 'DTI Ratio (%)', 'Savings ($)']
X = customers[feature_columns]
y = customers['Loan Status']
new_applicant = pd.DataFrame({'Credit Score': [630], 'DTI Ratio (%)': [40], 'Savings ($)': [2500]})
customers

,Customer ID,Credit Score,DTI Ratio (%),Savings ($),Loan Status
0,C1,780,12,25000,0
1,C2,720,18,15000,0
2,C3,690,25,8000,0
3,C4,660,32,5000,0
4,C5,640,38,3000,0
5,C6,610,42,2000,1
6,C7,580,48,1200,1
7,C8,550,55,800,1
8,C9,520,60,400,1
9,C10,480,68,100,1


## Train and evaluate the Decision Tree

In [4]:
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X, y)
predicted_status = int(tree_model.predict(new_applicant)[0])
pd.DataFrame({
    'Algorithm': ['Decision Tree'],
    'Predicted Status': [predicted_status],
    'Classification': ['Default / High Risk' if predicted_status else 'Approved / Low Risk'],
})

,Algorithm,Predicted Status,Classification
0,Decision Tree,0,Approved / Low Risk


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(tree_model, X, y, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'])
metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1'],
    'Mean': [scores['test_accuracy'].mean(), scores['test_precision'].mean(), scores['test_recall'].mean(), scores['test_f1'].mean()],
    'Std': [scores['test_accuracy'].std(), scores['test_precision'].std(), scores['test_recall'].std(), scores['test_f1'].std()],
}).round(3)
metrics_table

,Metric,Mean,Std
0,Accuracy,1.0,0.0
1,Precision,1.0,0.0
2,Recall,1.0,0.0
3,F1,1.0,0.0


## Predict applicants from a local CSV


Required first three columns: `Credit Score`, `DTI Ratio (%)`, `Savings ($)`. Add an optional `Loan Status` column with 0 or 1 to calculate accuracy, precision, recall, and F1.


In [6]:
def validate_csv_dataframe(csv_dataframe):
    if csv_dataframe.empty:
        raise ValueError('The CSV file is empty.')
    if list(csv_dataframe.columns[:3]) != feature_columns:
        raise ValueError('The first three columns must be exactly: ' + ', '.join(feature_columns))
    prepared = csv_dataframe.copy()
    for column in feature_columns:
        prepared[column] = pd.to_numeric(prepared[column], errors='coerce')
    if prepared[feature_columns].isna().any().any():
        raise ValueError('Feature columns must contain only numeric values with no blanks.')
    if 'Loan Status' in prepared.columns:
        prepared['Loan Status'] = pd.to_numeric(prepared['Loan Status'], errors='coerce')
        if prepared['Loan Status'].isna().any() or not prepared['Loan Status'].isin([0, 1]).all():
            raise ValueError('Loan Status must contain only 0 or 1 when provided.')
        prepared['Loan Status'] = prepared['Loan Status'].astype(int)
    return prepared

def predict_uploaded_csv(csv_dataframe):
    prepared = validate_csv_dataframe(csv_dataframe)
    predicted_status = tree_model.predict(prepared[feature_columns])
    predictions = prepared.copy()
    predictions['Decision Tree Prediction'] = predicted_status
    display(predictions)
    if 'Loan Status' in prepared.columns:
        actual_status = prepared['Loan Status']
        display(pd.DataFrame([{
            'Algorithm': 'Decision Tree',
            'Accuracy': accuracy_score(actual_status, predicted_status),
            'Precision': precision_score(actual_status, predicted_status, zero_division=0),
            'Recall': recall_score(actual_status, predicted_status, zero_division=0),
            'F1': f1_score(actual_status, predicted_status, zero_division=0),
        }]).round(3))
    else:
        print('No Loan Status column found; prediction metrics were not calculated.')

csv_uploader = widgets.FileUpload(accept='.csv', multiple=False)
display(csv_uploader)

def handle_csv_upload(_change):
    if not csv_uploader.value:
        return
    upload_value = csv_uploader.value
    uploaded_file = next(iter(upload_value.values())) if isinstance(upload_value, dict) else next(iter(upload_value))
    uploaded_metadata = uploaded_file.get('metadata', {})
    uploaded_name = uploaded_file.get('name') or uploaded_metadata.get('name') or 'uploaded.csv'
    print(f'Loaded: {uploaded_name}')
    predict_uploaded_csv(pd.read_csv(io.BytesIO(bytes(uploaded_file['content']))))

csv_uploader.observe(handle_csv_upload, names='value')

FileUpload(value={}, accept='.csv', description='Upload')

,Credit Score,DTI Ratio (%),Savings ($),Unnamed: 3,Customer ID,Loan Status,Decision Tree Prediction
0,780,12,25000,0,C1,0,0
1,720,18,15000,1,C2,0,0
2,690,25,8000,2,C3,0,0
3,660,32,5000,3,C4,0,0
4,640,38,3000,4,C5,0,0
5,610,42,2000,5,C6,1,1
6,580,48,1200,6,C7,1,1
7,550,55,800,7,C8,1,1
8,520,60,400,8,C9,1,1
9,480,68,100,9,C10,1,1


,Algorithm,Accuracy,Precision,Recall,F1
0,Decision Tree,1.0,1.0,1.0,1.0


### Summary
- This notebook is dedicated to the Decision Tree model.
- The built-in dataset and uploaded CSVs use the same three features.
- Metrics are calculated only when labeled records include `Loan Status`.